In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================================
# TRANSITION VERS LE NOTEBOOK 03
# ============================================================================

print("\n" + "="*60)
print("🎉 NOTEBOOK 02 - COMPLÈTEMENT TERMINÉ !")
print("="*60)

print("""
╔═══════════════════════════════════════════════════════════════════════════════╗
║                           BILAN DU PRETRAITEMENT                              ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  ✅ Filtrage : 1-40 Hz + Notch 50 Hz                                          ║
║  ✅ Référence : Moyenne                                                        ║
║  ✅ ICA : 20 composantes (suppression artefacts)                              ║
║  ✅ Epoching : 87 → {summary['n_epochs_final']} epochs                         ║
║  ✅ Classe équilibrée : G={summary['n_main_gauche']} D={summary['n_main_droite']}║
║                                                                               ║
║  📁 Données prêtes : reports/epochs_clean.fif                                 ║
║                                                                               ║
╚═══════════════════════════════════════════════════════════════════════════════╝
""")

print("\n➡️ PROCHAIN NOTEBOOK : 03_visualization.ipynb")
print("   ┌─────────────────────────────────────────────────────────────────┐")
print("   │ Objectifs :                                                      │")
print("   │   - Time-Frequency Representation (TFR) avec ondelettes Morlet   │")
print("   │   - Calcul ERD/ERS (Event-Related Desynchronization)             │")
print("   │   - Topomaps par bande de fréquence (alpha, beta, gamma)         │")
print("   │   - Butterfly plots et comparaison inter-conditions              │")
print("   │   - Visualisation de la latéralisation hémisphérique             │")
print("   └─────────────────────────────────────────────────────────────────┘")

print("\n💾 Pour continuer, assure-toi que ces fichiers existent :")
print("   - reports/epochs_clean.fif")
print("   - reports/evoked_left.fif")
print("   - reports/evoked_right.fif")


🎉 NOTEBOOK 02 - COMPLÈTEMENT TERMINÉ !

╔═══════════════════════════════════════════════════════════════════════════════╗
║                           BILAN DU PRETRAITEMENT                              ║
╠═══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  ✅ Filtrage : 1-40 Hz + Notch 50 Hz                                          ║
║  ✅ Référence : Moyenne                                                        ║
║  ✅ ICA : 20 composantes (suppression artefacts)                              ║
║  ✅ Epoching : 87 → {summary['n_epochs_final']} epochs                         ║
║  ✅ Classe équilibrée : G={summary['n_main_gauche']} D={summary['n_main_droite']}║
║                                                                               ║
║  📁 Données prêtes : reports/epochs_clean.fif                                 ║
║                                                         

In [3]:
!pip install mne -q

import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
mne.set_log_level('WARNING')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Création des dossiers
Path("figures").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

print("✅ Bibliothèques chargées")
print(f"MNE version: {mne.__version__}")

✅ Bibliothèques chargées
MNE version: 1.11.0


In [4]:
# ============================================================================
# CHARGEMENT ET PRÉTRAITEMENT COMPLET
# ============================================================================

print("\n" + "="*60)
print("CHARGEMENT DU SUJET 1")
print("="*60)

from mne.datasets import eegbci

# Chargement
raw_fnames = eegbci.load_data(1, [6, 10, 14], verbose=False)
raws = []
for fname in raw_fnames:
    raw_temp = mne.io.read_raw_edf(fname, preload=True, verbose=False)
    raws.append(raw_temp)

raw = mne.concatenate_raws(raws, verbose=False)
mne.datasets.eegbci.standardize(raw)
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, verbose=False)

print(f"✅ Sujet chargé : {raw.times[-1]:.1f} secondes, {len(raw.ch_names)} canaux")

# Filtrage
print("\n📊 Filtrage...")
raw_filtered = raw.copy()
raw_filtered.filter(1.0, 40.0, method='fir', verbose=False)
raw_filtered.notch_filter(50.0, picks='eeg', method='fir', verbose=False)
print("✅ Filtrage 1-40Hz + Notch 50Hz appliqué")

# Référence
raw_referenced = raw_filtered.copy()
raw_referenced.set_eeg_reference('average', verbose=False)
print("✅ Average reference appliqué")

# Epoching
events, event_id = mne.events_from_annotations(raw_referenced, verbose=False)
print(f"\n🎯 Événements : {len(events)}")
print(f"   Classes : {event_id}")

tmin, tmax = -1.0, 4.0
epochs = mne.Epochs(
    raw_referenced, events, event_id,
    tmin=tmin, tmax=tmax, baseline=(-1.0, 0),
    picks='eeg', preload=True, verbose=False
)

print(f"\n✅ Epochs créées : {len(epochs)}")
print(f"   Shape : {epochs.get_data().shape}")

# Distribution
print(f"\n📊 Distribution :")
for name, code in event_id.items():
    n = np.sum(epochs.events[:, -1] == code)
    print(f"   - {name} : {n}")

# Séparer les classes
epochs_left = epochs['T1']
epochs_right = epochs['T2']

print(f"\n✅ Prétraitement terminé !")
print(f"   - Main gauche : {len(epochs_left)} epochs")
print(f"   - Main droite : {len(epochs_right)} epochs")


CHARGEMENT DU SUJET 1


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
# ============================================================================
# ERPs SUR C3 ET C4
# ============================================================================

print("\n" + "="*60)
print("ERPs - CANAUX MOTEURS")
print("="*60)

evoked_left = epochs_left.average()
evoked_right = epochs_right.average()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mne.viz.plot_compare_evokeds(
    {'Main Gauche': evoked_left, 'Main Droite': evoked_right},
    picks=['C3'], axes=axes[0], show=False, title='Canal C3'
)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5, label='Début imagerie')
axes[0].grid(True, alpha=0.3)

mne.viz.plot_compare_evokeds(
    {'Main Gauche': evoked_left, 'Main Droite': evoked_right},
    picks=['C4'], axes=axes[1], show=False, title='Canal C4'
)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.suptitle('ERPs - Comparaison Main Gauche vs Main Droite', fontsize=14)
plt.tight_layout()
plt.show()

print("✅ Visualisation ERP affichée")

In [ ]:
print("""
🧠 INTERPRÉTATION NEUROLOGIQUE — ERPs :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• C3 (hémisphère gauche) contrôle la main DROITE
• C4 (hémisphère droit)  contrôle la main GAUCHE
• On attend une amplitude plus négative sur C3 pour T2
  (main droite) et sur C4 pour T1 (main gauche)
• Ce croisement est la signature de la latéralisation
  motrice controlatérale — base neurologique du BCI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")


In [ ]:
# ============================================================================
# TOPOMAPS PAR BANDE DE FRÉQUENCE
# ============================================================================

print("\n" + "="*60)
print("TOPOMAPS PAR BANDE DE FRÉQUENCE")
print("="*60)

bands = {
    'Delta (0.5-4 Hz)': (0.5, 4),
    'Theta (4-8 Hz)': (4, 8),
    'Alpha (8-13 Hz)': (8, 13),
    'Beta (13-30 Hz)': (13, 30),
    'Gamma (30-45 Hz)': (30, 45)
}

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Topomaps par bande - Main Gauche vs Main Droite', fontsize=16)

for idx, (band_name, (fmin, fmax)) in enumerate(bands.items()):
    # Filtrer
    epochs_left_band = epochs_left.copy().filter(fmin, fmax, verbose=False)
    epochs_right_band = epochs_right.copy().filter(fmin, fmax, verbose=False)
    
    # Puissance moyenne sur période d'imagerie (0-3s)
    evoked_left_band = epochs_left_band.average()
    evoked_right_band = epochs_right_band.average()
    
    times_idx = (evoked_left_band.times >= 0) & (evoked_left_band.times <= 3)
    power_left = np.mean(evoked_left_band.data[:, times_idx], axis=1)
    power_right = np.mean(evoked_right_band.data[:, times_idx], axis=1)
    
    # Topomaps
    mne.viz.plot_topomap(power_left, evoked_left_band.info, axes=axes[0, idx], 
                          show=False, names=epochs_left.ch_names)
    axes[0, idx].set_title(f'Main Gauche\n{band_name}', fontsize=9)
    
    mne.viz.plot_topomap(power_right, evoked_right_band.info, axes=axes[1, idx],
                          show=False, names=epochs_right.ch_names)
    axes[1, idx].set_title(f'Main Droite\n{band_name}', fontsize=9)

plt.tight_layout()
plt.savefig('figures/topomaps_by_band.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : figures/topomaps_by_band.png")

In [ ]:
print("""
🧠 INTERPRÉTATION NEUROLOGIQUE — Topomaps par bande :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Delta/Theta : activité diffuse, peu informative pour la motricité
• Alpha (8-13 Hz) : désynchronisation occipitale au repos
• Beta (13-30 Hz) : ERD (diminution de puissance) controlatérale
  au mouvement imaginé — zone centrale C3/C4
• La latéralisation visible sur les topomaps beta CONFIRME
  que le signal encode l'intention motrice de façon spatiale
• Un classifieur BCI exploite précisément cette asymétrie C3/C4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")


In [ ]:
# ============================================================================
# TFR - ONDELETTES DE MORLET
# ============================================================================

print("\n" + "="*60)
print("TFR - ONDELETTES DE MORLET")
print("="*60)

from mne.time_frequency import tfr_morlet

freqs = np.arange(6, 31, 1)
n_cycles = freqs / 2.0

print("🔄 Calcul TFR (peut prendre 30-60 secondes)...")

tfr_left = tfr_morlet(epochs_left, freqs=freqs, n_cycles=n_cycles,
                       use_fft=True, return_itc=False, decim=3, verbose=False)
tfr_right = tfr_morlet(epochs_right, freqs=freqs, n_cycles=n_cycles,
                        use_fft=True, return_itc=False, decim=3, verbose=False)

tfr_left.apply_baseline(baseline=(-1, 0), mode='percent')
tfr_right.apply_baseline(baseline=(-1, 0), mode='percent')

print("✅ TFR calculés")

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tfr_left.plot(picks=['C3'], axes=axes[0, 0], show=False)
axes[0, 0].set_title('Canal C3 - Main Gauche', fontsize=12)
axes[0, 0].axvline(0, color='white', linestyle='--')

tfr_right.plot(picks=['C3'], axes=axes[0, 1], show=False)
axes[0, 1].set_title('Canal C3 - Main Droite', fontsize=12)
axes[0, 1].axvline(0, color='white', linestyle='--')

tfr_left.plot(picks=['C4'], axes=axes[1, 0], show=False)
axes[1, 0].set_title('Canal C4 - Main Gauche', fontsize=12)
axes[1, 0].axvline(0, color='white', linestyle='--')

tfr_right.plot(picks=['C4'], axes=axes[1, 1], show=False)
axes[1, 1].set_title('Canal C4 - Main Droite', fontsize=12)
axes[1, 1].axvline(0, color='white', linestyle='--')

plt.suptitle('TFR Morlet (6-30 Hz) - Comparaison Main Gauche vs Main Droite', fontsize=14)
plt.tight_layout()
plt.savefig('figures/tfr_morlet.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Figure sauvegardée : figures/tfr_morlet.png")

In [ ]:
print("""
🧠 INTERPRÉTATION NEUROLOGIQUE — TFR Morlet (ERD/ERS) :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Zone BLEUE après t=0 → ERD : le cerveau DÉSACTIVE la bande beta
  ipsilatérale pour libérer le circuit moteur controlatéral
• Zone ROUGE après t=0 → ERS : rebond post-mouvement (resynchro)
• Sur C3 : ERD plus marqué pour T2 (main droite) car C3 = hémisphère gauche
• Sur C4 : ERD plus marqué pour T1 (main gauche) car C4 = hémisphère droit
• L'axe temporel montre que l'activation commence AVANT t=0
  → le cerveau anticipe l'intention motrice (potentiel BCI)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")


In [ ]:
# ============================================================================
# BUTTERFLY PLOTS (CORRIGÉ)
# ============================================================================

print("\n" + "="*60)
print("BUTTERFLY PLOTS")
print("="*60)

# Butterfly plot pour Main Gauche
fig = evoked_left.plot(picks='eeg', gfp=True, show=False)
fig.canvas.manager.set_window_title('Butterfly plot - Main Gauche')
plt.savefig('figures/butterfly_left.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# Butterfly plot pour Main Droite
fig = evoked_right.plot(picks='eeg', gfp=True, show=False)
fig.canvas.manager.set_window_title('Butterfly plot - Main Droite')
plt.savefig('figures/butterfly_right.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

print("✅ Butterfly plots sauvegardés : figures/butterfly_left.png, figures/butterfly_right.png")

# ============================================================================
# RAPPORT FINAL
# ============================================================================

print("\n" + "="*60)
print("🎉 NOTEBOOK 03 - VISUALISATION TERMINÉE !")
print("="*60)

print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                        RÉSUMÉ DES VISUALISATIONS                           ║
╠═══════════════════════════════════════════════════════════════════════════╣
║                                                                           ║
║  ✅ ERPs sur C3 et C4 (potentiels évoqués)                                ║
║  ✅ Topomaps par bande (Delta, Theta, Alpha, Beta, Gamma)                 ║
║  ✅ TFR Morlet (6-30 Hz) sur canaux C3 et C4                              ║
║  ✅ Butterfly plots pour les deux conditions                              ║
║                                                                           ║
║  📁 Figures sauvegardées dans le dossier 'figures/'                       ║
║                                                                           ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")

print("\n📊 Interprétations clés pour soutenance :")
print("   1. ERD en bande bêta (13-30 Hz) sur cortex controlatéral")
print("   2. Latéralisation : main droite → C3, main gauche → C4")
print("   3. Topomaps alpha : forte puissance occipitale (repos visuel)")
print("   4. TFR confirme la désynchronisation autour de t=0")

print("\n📁 Fichiers générés dans 'figures/' :")
print("   - topomaps_by_band.png")
print("   - tfr_morlet.png")
print("   - butterfly_left.png")
print("   - butterfly_right.png")

print("\n➡️ Prochain notebook : 04_augmentation (données synthétiques)")